## Optimized parameters : 

#### Constant Weights Genome

The _progress weight_ evaluates how valuable temporary progress is in a column when deciding whether to stop :

$w^{(\mathrm{progress})}_i=1+\alpha \, |7-i|$ 
- $i$ being the number of the column
- $\alpha$ being the `outer_progress_bonus`

The _move weight_ is used to score candidate dice pairings.

$w^{(\mathrm{move})}_i=1+\beta \, (5 - |7-i|)$
- $i$ being the number of the column
- $\beta$ being the `center_move_bonus`

If a move $m$ advances the columns contained in the set $C(m)$, then its _move score_ is:

$S_{\mathrm{move}}(m) = \sum_{i \in C(m)} w^{(\mathrm{move})}_i - k \sum_{i \in C(m)} \mathbf{1}_{i \notin N}$
- $N$ is the set of currently active neutral-marker columns
- $k$ is the `marker_penalty`

The chosen move is: $m^\star = \arg\max_m S_{\mathrm{move}}(m)$

The _temporary progress_ score used to decide whether to stop is:

$S_{\mathrm{progress}} = \sum_{i \in N} (p_i+1) w^{(\mathrm{progress})}_i + B(N)$

- $N$ is the set of active neutral-marker columns
- $p_i$ is the temporary progress made during the current turn in column i
- $B(N)$ is a _bonus/penalty_ depending on the parity of the active columns.

$
B(N)=
\begin{cases}
o & \text{if all columns in } N \text{ are odd} \\
e & \text{if all columns in } N \text{ are even} \\
0 & \text{otherwise}
\end{cases}
$

- $o$ corresponds to `odd_bonus`
- $e$ corresponds to `even_bonus`

The _stopping rule_ is:

$\text{Stop if } S_{\mathrm{progress}} \ge t$

where t is the `threshold`.

In the paper _OPTIMIZING GENETIC ALGORITHM PARAMETERS FOR A
STOCHASTIC GAME_, they chose : 

$
p_i \in \{0,\dots,7\}
$

$
b_i \in \{0,\dots,7\}
$

$
e,o,h \in \{-8,\dots,7\}
$

$
k \in \{0,\dots,15\}
$

$
t \in \{0,\dots,31\}
$

$
m_i \in \mathcal{M}
$

where $\mathcal{M}$ is a set of $32$ values chosen between $0$ and $64$.

In [1]:
import random
import numpy as np

COLUMNS = [i for i in range(2, 13)]
HEIGHT = {2:3, 3:5, 4:7, 5:9, 6:11, 7:13, 8:11, 9:9, 10:7, 11:5, 12:3}

# Genome
def random_genome():
    "1 candidate strategy, contains the parameters we want to optimize, sets them randomly"
    return {
        "outer_progress_bonus": random.uniform(0.2, 2.0),
        "center_move_bonus": random.uniform(0.2, 2.0),
        "odd_bonus": random.uniform(-5, 8),
        "even_bonus": random.uniform(-5, 8),
        "marker_penalty": random.uniform(0, 10),
        "threshold": random.uniform(15, 40),
    }

def progress_weight(col, genome):
    return 1 + genome["outer_progress_bonus"] * abs(7 - col)

def move_weight(col, genome):
    return 1 + genome["center_move_bonus"] * (5 - abs(7 - col))

In [14]:
# Dice pairings
def possible_pairings(dice):
    "returns a list of sets of possible pairings"
    a, b, c, d = dice
    pairings = [(a+b, c+d), (a+c, b+d), (a+d, b+c)]
    return list(set(tuple(sorted(p)) for p in pairings))

In [6]:
# Rules of the game
def can_use_column(col, permanent, neutral):
    "checks whether the player can advance up the column"
    if permanent[col] >= HEIGHT[col]:
        return False
    return col in neutral or len(neutral) < 3

def legal_moves(dice, permanent, neutral):
    "Computes all legal moves associated with a dice roll"
    moves = []

    for p1, p2 in possible_pairings(dice):
        cols = [p1, p2]

        new_cols = [c for c in cols if c not in neutral]
        if len(neutral) + len(set(new_cols)) > 3:
            usable = [c for c in cols if c in neutral and permanent[c] < HEIGHT[c]]
        else:
            usable = [c for c in cols if can_use_column(c, permanent, neutral)]

        if len(usable) == 2:
            moves.append(tuple(cols))
        elif len(usable) == 1:
            moves.append((usable[0],))

    return list(set(moves))

def apply_move(move, permanent, neutral):
    "Applies a move by advancing the temporary neutral markers"
    neutral = neutral.copy()

    for col in move:
        if col not in neutral:
            neutral[col] = permanent[col]
        if neutral[col] < HEIGHT[col]:
            neutral[col] += 1

    return neutral

In [7]:
# Heuristic policy

def score_move(move, permanent, neutral, genome):
    """
    Scores a candidate move using the heuristic:
    rewarding valuable columns and penalizing new markers
    """
    score = 0
    
    for col in move:
        score += move_weight(col, genome)
        if col not in neutral:
            score -= genome["marker_penalty"]
    return score
    

def choose_move(moves, permanent, neutral, genome):
    "Selects the move with the highest heuristic score"
    return max(moves, key=lambda move: score_move(move, permanent, neutral, genome))


def progress_score(permanent, neutral, genome):
    "Computes the temporary progress score used to decide whether the player should stop"
    score = 0
    active_cols = list(neutral.keys())

    for col in active_cols:
        progress = neutral[col] - permanent[col]
        score += (progress + 1) * progress_weight(col, genome)

    if len(active_cols) == 3:
        if all(c % 2 == 1 for c in active_cols):
            score += genome["odd_bonus"]
        if all(c % 2 == 0 for c in active_cols):
            score += genome["even_bonus"]
    return score


def should_stop(permanent, neutral, genome):
    "Returns True if the progress score is large enough to stop and bank the current progress, False otherwise"
    return progress_score(permanent, neutral, genome) >= genome["threshold"]

In [8]:
# Simulation

def simulate_game(genome, max_turns=200):
    "Simulates one full solitaire game of Can't Stop using the strategy encoded in the genome"
    permanent = {c: 0 for c in COLUMNS}
    turns = 0

    while turns < max_turns:
        turns += 1
        neutral = {}

        while True:
            dice = [random.randint(1, 6) for _ in range(4)]
            moves = legal_moves(dice, permanent, neutral)

            if not moves:
                break  # bust: loose neutral progress

            move = choose_move(moves, permanent, neutral, genome)
            neutral = apply_move(move, permanent, neutral)

            won_cols = sum(permanent[c] >= HEIGHT[c] for c in COLUMNS)
            won_cols += sum(neutral[c] >= HEIGHT[c] and permanent[c] < HEIGHT[c] for c in neutral)

            if won_cols >= 3:
                return turns

            if should_stop(permanent, neutral, genome):
                for col, pos in neutral.items():
                    permanent[col] = max(permanent[col], pos)
                break

        if sum(permanent[c] >= HEIGHT[c] for c in COLUMNS) >= 3:
            return turns

    return max_turns


def evaluate_genome(genome, n_games=100):
    "Evaluates a genome by simulating several games and averaging the number of turns needed to win"
    scores = [simulate_game(genome) for _ in range(n_games)]
    return np.mean(scores)

In [9]:
# Genetic Algorithm

def tournament_selection(population, fitnesses, k=3):
    """
    Selects a parent genome using tournament selection:
    several genomes are sampled randomly and the best is kept
    """
    candidates = random.sample(list(zip(population, fitnesses)), k)
    return min(candidates, key=lambda x: x[1])[0]  # lower turns is better


def crossover(parent1, parent2):
    """
    Creates a child genome by randomly mixing
    parameters from two parent genomes.
    """
    child = {}

    for key in parent1:
        if random.random() < 0.5:
            child[key] = parent1[key]
        else:
            child[key] = parent2[key]

    return child


def mutate(genome, mutation_rate=0.2):
    """
    Randomly perturbs (with gaussian noise) some parameters of a genome
    in order to maintain diversity in the population.
    """
    child = genome.copy()

    ranges = {
        "outer_progress_bonus": (0.2, 2.0),
        "center_move_bonus": (0.2, 2.0),
        "odd_bonus": (-5, 8),
        "even_bonus": (-5, 8),
        "marker_penalty": (0, 10),
        "threshold": (15, 40),
    }

    for key, (low, high) in ranges.items():
        if random.random() < mutation_rate:
            noise = random.gauss(0, 0.15 * (high - low))
            child[key] += noise
            child[key] = max(low, min(high, child[key]))

    return child


def genetic_algorithm(population_size=50, generations=30, games_per_genome=100, elite_size=5):
    """
    Main GA loop: evaluate genomes, keep the best ones,
    generate children through crossover and mutation, and repeat over several generations.
    """
    population = [random_genome() for _ in range(population_size)]
    best_history = []

    for gen in range(generations):
        fitnesses = [evaluate_genome(g, n_games=games_per_genome) for g in population]

        ranked = sorted(zip(population, fitnesses), key=lambda x: x[1])
        best_genome, best_fitness = ranked[0]
        best_history.append(best_fitness)

        print(f"Generation {gen:02d} | best avg turns = {best_fitness:.3f}")
        print(f"Generation {gen:02d} | best avg turns = {best_fitness:.3f}")
        pretty_genome = {k: round(v, 3) for k, v in best_genome.items()}
        print(pretty_genome)

        new_population = [g for g, f in ranked[:elite_size]]

        while len(new_population) < population_size:
            p1 = tournament_selection(population, fitnesses)
            p2 = tournament_selection(population, fitnesses)

            child = crossover(p1, p2)
            child = mutate(child)
 
            new_population.append(child)

        population = new_population

    final_fitnesses = [evaluate_genome(g, n_games=games_per_genome * 5) for g in population ]

    ranked = sorted(zip(population, final_fitnesses), key=lambda x: x[1])
    return ranked[0], best_history

In [10]:
(best_genome, best_score), history = genetic_algorithm(
    population_size=50,
    generations=20,
    games_per_genome=100,
)

print("Best genome:")
print(best_genome)

print("Estimated average turns:")
print(best_score)

Generation 00 | best avg turns = 11.500
Generation 00 | best avg turns = 11.500
{'outer_progress_bonus': 0.779, 'center_move_bonus': 0.359, 'odd_bonus': -3.671, 'even_bonus': 0.395, 'marker_penalty': 9.706, 'threshold': 25.439}
Generation 01 | best avg turns = 11.930
Generation 01 | best avg turns = 11.930
{'outer_progress_bonus': 1.241, 'center_move_bonus': 1.708, 'odd_bonus': 0.925, 'even_bonus': -3.436, 'marker_penalty': 5.423, 'threshold': 31.196}
Generation 02 | best avg turns = 11.660
Generation 02 | best avg turns = 11.660
{'outer_progress_bonus': 1.241, 'center_move_bonus': 1.708, 'odd_bonus': 0.925, 'even_bonus': -3.436, 'marker_penalty': 5.423, 'threshold': 31.196}
Generation 03 | best avg turns = 11.370
Generation 03 | best avg turns = 11.370
{'outer_progress_bonus': 1.598, 'center_move_bonus': 0.609, 'odd_bonus': -4.627, 'even_bonus': 3.211, 'marker_penalty': 2.059, 'threshold': 36.372}
Generation 04 | best avg turns = 11.590
Generation 04 | best avg turns = 11.590
{'outer_

The best evolved strategies consistently achieved around 11.2–11.5 turns on average.

The evolved threshold converged near 30, which is very close to the original Rule of 28 stopping value = 28
 
The algorithm learned that progress in outer columns is valuable: `outer_progress_bonus` around 1

The evolved strategies strongly preferred central columns when choosing moves: `center_move_bonus` around 1.5–2

Odd-only active configurations were generally considered risky: `odd_bonus`>0

Even-only active configurations were generally considered safer: `even_bonus`<0

The evolved strategies learned a moderate penalty for opening new neutral markers: `marker_penalty` around 5


the heuristic uses only 6 parameters,
the game simulation is simplified,
the training budget is relatively small.

## Compared to the paper :

In [ ]:
The paper gets

even_bonus = 1
odd_bonus = 7
high_bonus = 6
low_bonus = 5
marker_penalty = 6
threshold = 29

But it could be because the paper adds different move & progress weights on each column :

"progress_weights": {2:p2, 3:p3, 4:p4, 5:p5, 6:p6, 7:p7, 8:p6, 9:p5, 10:p4, 11:p3, 12:p2}

"move_weights": {2:v2, 3:v3, 4:v4, 5:v5, 6:v6, 7:v7, 8:v6, 9:v5, 10:v4, 11:v3, 12:v2}

to the genome.